# Intégration LFM2.5-Audio dans vLLM-Omni

Sert le modèle **LiquidAI/LFM2.5-Audio** (S2S interleaved texte+audio) via **vLLM-Omni** 
pour le prefix caching, le paged KV et le continuous batching — gains visés sur le 
round-trip tool-calling et le multi-sessions.

Le plugin out-of-tree vit dans le repo `rcarvalo/finetuning_s2s_toolcalling` 
(branche `claude/blissful-tesla-7i1yky`), package `vllm_omni_lfm2_audio`. Ce notebook le clone, l'installe, 
convertit le checkpoint, et valide la chaîne de bout en bout.

> **Runtime requis : GPU** (Exécution → Modifier le type d'exécution → T4/L4/A100). 
> L'engine `vllm-omni` est CUDA-only — il ne tourne pas sur CPU/Mac.

Chaque cellule encode un écart runtime déjà diagnostiqué (vllm-omni 0.22.0) — exécute-les dans l'ordre.

## 1. Vérification du GPU

In [3]:
import sys
print('Python', sys.version.split()[0])
!nvidia-smi -L || echo 'AUCUN GPU — changer le type d execution avant de continuer'

Python 3.12.13
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-c46ccdc2-6771-1d89-31b6-6b9abf66e465)


## 2. Clone du repo (plugin)

Idempotent : re-exécutable sans risque (pull si déjà cloné).

In [4]:
import os
REPO = 'https://github.com/rcarvalo/finetuning_s2s_toolcalling.git'
BRANCH = 'claude/blissful-tesla-7i1yky'
DST = '/content/finetuning_s2s_toolcalling'
if not os.path.isdir(DST):
    !git clone -q -b {BRANCH} {REPO} {DST}
%cd {DST}
!git pull --ff-only -q && git log --oneline -1

/content/finetuning_s2s_toolcalling
24a01ce (HEAD -> claude/blissful-tesla-7i1yky, origin/claude/blissful-tesla-7i1yky, origin/HEAD) notebook Colab complet d'intégration LFM2.5-Audio dans vLLM-Omni


## 3. Installation des dépendances

**Écart 1** — `vllm-omni` ne déclare PAS `vllm` en dépendance : on l'installe séparément 
à la version appariée (`0.22.0`). On utilise `sys.executable -m pip` (le `!pip` de Colab 
peut viser un autre interpréteur que le kernel). Compter ~6-8 min sur VM fraîche.

In [5]:
import sys
!{sys.executable} -m pip install -q 'vllm==0.22.0' 'vllm-omni==0.22.0' 'liquid-audio>=1.3.0'
!{sys.executable} -m pip install -q -e /content/finetuning_s2s_toolcalling --no-deps
import importlib.metadata as md_
print('vllm', md_.version('vllm'), '| vllm-omni', md_.version('vllm-omni'), '| liquid-audio', md_.version('liquid-audio'))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for s2s-toolcalling (pyproject.toml) ... done
vllm 0.22.0 | vllm-omni 0.22.0 | liquid-audio 1.3.0


## 4. Correctif des libs CUDA (écart 2)

Le wheel `vllm` 0.22.0 est buildé pour **CUDA 13** alors que le torch de Colab est en cu12x : 
`import vllm_omni` échoue sur `libcudart.so.13` introuvable. On précharge les libs `cu13` 
(ctypes, pour ce kernel) **et** on exporte `LD_LIBRARY_PATH` (hérité par les sous-process 
`StageEngineCoreProc` que l'engine spawn). À exécuter **avant** tout import de `vllm_omni`.

In [6]:
import os, glob, ctypes
cu13_dirs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cu13/lib')
assert cu13_dirs, 'paquet nvidia cu13 introuvable (réexécuter la cellule 3 ?)'
cu13 = cu13_dirs[0]
os.environ['LD_LIBRARY_PATH'] = cu13 + ':' + os.environ.get('LD_LIBRARY_PATH', '')
for so in sorted(glob.glob(cu13 + '/lib*.so*')):
    try:
        ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass
import vllm, vllm_omni
print('vllm', vllm.__version__, '| vllm_omni', vllm_omni.__version__, '— imports OK')

WARNING 06-11 14:58:18 [patch.py:165] Monkey-patched unregister_vllm_metrics() to scope drops to non-omni vllm:* collectors. Remove this patch once vLLM adds _STAT_LOGGER_METRIC_NAMES.
vllm 0.22.0 | vllm_omni 0.22.0 — imports OK


## 5. Smoke-test progressif (imports → plugin → contrat runtime)

Vérifie que l'entry point `lfm2_audio` est découvert, que l'architecture + le pipeline 
sont enregistrés, et que le contrat vllm-omni 0.22.0 (hook `sample()`, champs 
`StagePipelineConfig`) est présent.

In [7]:
import sys
!cd /content/finetuning_s2s_toolcalling && {sys.executable} scripts/colab_smoke_vllm_omni.py

[ ok ] imports: torch 2.11.0+cu128
[ ok ] imports: vllm 0.22.0
WARNING 06-11 14:58:34 [patch.py:165] Monkey-patched unregister_vllm_metrics() to scope drops to non-omni vllm:* collectors. Remove this patch once vLLM adds _STAT_LOGGER_METRIC_NAMES.
[ ok ] imports: vllm_omni 0.22.0
[ ok ] imports: liquid_audio ?
[ ok ] imports: CUDA 12.8 — NVIDIA A100-SXM4-40GB
[ ok ] plugin: entry point 'lfm2_audio' découvert (vllm_omni_lfm2_audio:register)
[ ok ] plugin: load_omni_general_plugins() exécuté
[ ok ] plugin: pipeline 'lfm2_audio' présent dans le registre
[ ok ] plugin: architecture 'Lfm2AudioOmniModel' visible dans OmniModelRegistry
[ ok ] contract: OmniOutput('text_hidden_states', 'multimodal_outputs', 'intermediate_tensors', 'next_token_id')
[ ok ] contract: hook sample()/prefer_model_sampler présent dans GPUARModelRunner._sample
[ ok ] contract: tous les champs StagePipelineConfig utilisés par pipeline.py existent

RÉSULTAT : OK (imports, plugin, contract)


## 6. Conversion du checkpoint

Télécharge la base `LiquidAI/LFM2.5-Audio-1.5B` et la convertit au layout vLLM-Omni 
(`config.json` : `model_type=lfm2_audio`, architecture, ids placeholders, ratio interleaved). 
Pour servir TON modèle finetuné FR : remplace `BASE` par le chemin de ton export `--mode full`.

In [8]:
import sys, os
from huggingface_hub import snapshot_download
BASE = snapshot_download('LiquidAI/LFM2.5-Audio-1.5B')
OMNI = '/content/lfm25_audio_omni'
if not os.path.exists(os.path.join(OMNI, 'config.json')):
    !cd /content/finetuning_s2s_toolcalling && {sys.executable} -m vllm_omni_lfm2_audio.convert_checkpoint --checkpoint {BASE} --output {OMNI}
!cd /content/finetuning_s2s_toolcalling && {sys.executable} scripts/colab_smoke_vllm_omni.py --checkpoint {OMNI}

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[ ok ] imports: torch 2.11.0+cu128
[ ok ] imports: vllm 0.22.0
WARNING 06-11 14:58:53 [patch.py:165] Monkey-patched unregister_vllm_metrics() to scope drops to non-omni vllm:* collectors. Remove this patch once vLLM adds _STAT_LOGGER_METRIC_NAMES.
[ ok ] imports: vllm_omni 0.22.0
[ ok ] imports: liquid_audio ?
[ ok ] imports: CUDA 12.8 — NVIDIA A100-SXM4-40GB
[ ok ] plugin: entry point 'lfm2_audio' découvert (vllm_omni_lfm2_audio:register)
[ ok ] plugin: load_omni_general_plugins() exécuté
[ ok ] plugin: pipeline 'lfm2_audio' présent dans le registre
[ ok ] plugin: architecture 'Lfm2AudioOmniModel' visible dans OmniModelRegistry
[ ok ] contract: OmniOutput('text_hidden_states', 'multimodal_outputs', 'intermediate_tensors', 'next_token_id')
[ ok ] contract: hook sample()/prefer_model_sampler présent dans GPUARModelRunner._sample
[ ok ] contract: tous les champs StagePipelineConfig utilisés par pipeline.py existent
[ ok ] checkpoint: config OK (ratio 6:12)
[ ok ] checkpoint: 931 tenseurs

## 7. Démarrage de l'engine + génération texte (E2E)

Lance les 2 stages (AR interleaved → détokeniseur) et génère. Flags issus de l'itération :

- **écart 3** : `load_omni_general_plugins()` AVANT `Omni()` (la détection de pipeline 
  précède le chargement des plugins par l'engine) ;
- **écart 4** : un `SamplingParams` PAR stage (`num_stages`) ;
- **écart 5** : `async_scheduling=False` (le scheduling async tronque l'historique remis 
  au sampler custom → décale la machine à états interleaved d'un step) ;
- `enforce_eager=True`, `gpu_memory_utilization=0.42` (2 stages sur 1 GPU), `dtype=float16` 
  (T4 sans bf16 ; mettre `bfloat16` sur A100/L4 pour la parité), `stage_init_timeout=1200` (T4 lent).

Démarrage ~4-6 min (chargement fp16 des 2 stages + profiling).

In [9]:
import sys, os
SRC = '/content/finetuning_s2s_toolcalling/src'
sys.path.insert(0, SRC)                                          # ce kernel
os.environ['PYTHONPATH'] = SRC + ':' + os.environ.get('PYTHONPATH', '')  # workers spawn

import vllm_omni.plugins as _p
_p.omni_plugins_loaded = False        # un 1er essai a échoué → forcer le rechargement
import vllm_omni_lfm2_audio           # doit importer sans erreur maintenant

from vllm_omni.plugins import load_omni_general_plugins
load_omni_general_plugins()


In [12]:
import gc, torch
for n in ('model', 'proc', 'omni'):
    if n in globals():
        del globals()[n]
gc.collect(); torch.cuda.empty_cache()
print('VRAM libre:', torch.cuda.mem_get_info()[0] / 1e9, 'Go')


VRAM libre: 41.827106816 Go


In [ ]:
import sys, os
SRC = '/content/finetuning_s2s_toolcalling/src'
sys.path.insert(0, SRC)                                          # ce kernel
os.environ['PYTHONPATH'] = SRC + ':' + os.environ.get('PYTHONPATH', '')  # workers spawn
import vllm_omni.plugins as _p; _p.omni_plugins_loaded = False   # un essai a pu échouer
import vllm_omni_lfm2_audio                                      # doit importer
from vllm_omni.plugins import load_omni_general_plugins; load_omni_general_plugins()

from vllm import SamplingParams
from vllm_omni import Omni
from transformers import AutoTokenizer
OMNI = '/content/lfm25_audio_omni'
tok = AutoTokenizer.from_pretrained(OMNI)

def build_prompt_ids(user_text):
    text = ('<|startoftext|><|im_start|>system\n'
            'Respond with interleaved text and audio.<|im_end|>\n'
            f'<|im_start|>user\n{user_text}<|im_end|>\n'
            '<|im_start|>assistant\n')
    return tok(text, add_special_tokens=False).input_ids

# deploy_config = LE point clé : sans lui, ni connector streaming ni prefix caching
# (il porte async_chunk, dtype bfloat16, SharedMemoryConnector codec_streaming,
#  enable_prefix_caching, split mémoire par stage).
omni = Omni(
    model=OMNI,
    deploy_config='/content/finetuning_s2s_toolcalling/configs/vllm_omni_lfm2_audio.yaml',
    stage_init_timeout=1200,
    init_timeout=1800,
print('engine prêt — async_chunk:', omni.async_chunk)

INFO 06-11 15:01:06 [omni_base.py:166] [Omni] Initializing with model /content/lfm25_audio_omni
INFO 06-11 15:01:06 [async_omni_engine.py:294] [AsyncOmniEngine] Initializing with model /content/lfm25_audio_omni
WARNING 06-11 15:01:06 [stage_config.py:1267] Deploy config not found: /usr/local/lib/python3.12/dist-packages/vllm_omni/deploy/lfm2_audio.yaml — using pipeline defaults only
INFO 06-11 15:01:06 [async_omni_engine.py:364] [AsyncOmniEngine] Launching Orchestrator thread with 2 stages
INFO 06-11 15:01:06 [model.py:617] Resolved architecture: Lfm2AudioOmniModel
INFO 06-11 15:01:06 [model.py:1752] Using max model len 128000
WARNING 06-11 15:01:06 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-11 15:01:06 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-11 15:01:06 [

## 8. Génération interleaved : texte + frames audio

Greedy (parité), 2 `SamplingParams`. On inspecte les ids du stage 0 (placeholders frame/EOA = 
audio généré), la cadence, et on récupère le waveform du stage 1.

In [14]:
from vllm_omni_lfm2_audio.constants import (
    AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID, IM_END_TOKEN_ID)

prompt_ids = build_prompt_ids('Bonjour, qui es-tu ?')
sp0 = SamplingParams(temperature=0.0, max_tokens=128, stop_token_ids=[IM_END_TOKEN_ID])
sp1 = SamplingParams(max_tokens=1, detokenize=False)
outputs = omni.generate({'prompt_token_ids': prompt_ids}, [sp0, sp1])

wav = None
for out in outputs:
    ro = out.request_output
    if out.final_output_type == 'text' and ro is not None and ro.outputs:
        ids = list(ro.outputs[0].token_ids)
        n_frames = sum(i == AUDIO_FRAME_PLACEHOLDER_ID for i in ids)
        n_eoa = sum(i == AUDIO_EOA_PLACEHOLDER_ID for i in ids)
        text_ids = [i for i in ids if i not in (AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID)]
        print(f'stage 0 : {len(ids)} ids — {len(text_ids)} texte, {n_frames} frames, {n_eoa} EOA')
        print('texte :', repr(ro.outputs[0].text[:200]))
    elif out.final_output_type == 'audio':
        mm = getattr(ro, 'multimodal_output', None) or getattr(out, 'multimodal_output', None)
        print('stage 1 : sortie audio —', type(mm).__name__)
        wav = mm
print('frames audio générées' if n_frames else '[warn] aucune frame audio')

WARNING 06-11 15:02:16 [input_processor.py:274] Passing raw prompts to InputProcessor is deprecated and will be removed in v0.18. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().
INFO 06-11 15:02:16 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-0 [rep-0] add request: 0_fe3c0a93-c0f0-4125-864b-784a72420900
INFO 06-11 15:02:16 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-1 [rep-0] add request: 0_fe3c0a93-c0f0-4125-864b-784a72420900


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s]

stage 0 : 124 ids — 25 texte, 98 frames, 1 EOA
texte : 'I’m Liquid Lili, an audio‑first assistant built by Liquid AI. How can I help you today?'
stage 1 : sortie audio — dict
frames audio générées


## 9. Écoute du résultat

Reconstruit un tensor 1D depuis la sortie du stage 1 (12,5 frames/s → 24 kHz) et le joue. 
Si la structure de `multimodal_output` diffère, la cellule l'introspecte pour ajuster.

In [15]:
import numpy as np, torch
from IPython.display import Audio, display

def to_waveform(x):
    if x is None:
        return None
    if isinstance(x, dict):
        for k in ('model_outputs', 'audio', 'waveform', 'wav'):
            if k in x:
                return to_waveform(x[k])
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy().reshape(-1)
    if isinstance(x, np.ndarray):
        return x.reshape(-1)
    return None

w = to_waveform(wav)
if w is not None and w.size:
    print('waveform :', w.shape, f'({w.size/24000:.2f}s @ 24kHz)')
    display(Audio(w, rate=24000))
else:
    print('pas de waveform exploitable — structure brute :')
    print(repr(wav)[:500])

waveform : (407040,) (16.96s @ 24kHz)


## 10. Sonde automatique (optionnel)

Relance la chaîne via le script du repo (mêmes flags) avec un diagnostic compact — pratique 
pour itérer après un `git pull`. Redémarre l'engine dans un sous-process propre.

In [ ]:
import sys, os
env = dict(os.environ)
!cd /content/finetuning_s2s_toolcalling && LD_LIBRARY_PATH={env['LD_LIBRARY_PATH']} {sys.executable} scripts/colab_probe_interleaved.py --checkpoint /content/lfm25_audio_omni 2>&1 | tail -20

## Où on en est

| Étape | État |
|---|---|
| Plugin chargé, pipeline 2-stages enregistré | ✅ |
| Engine démarre, génère texte E2E | ✅ |
| Stage 0 : machine à états interleaved (sample/depthformer) | ✅ active en runtime |
| **Parité greedy vs `liquid_audio.generate_interleaved`** | ⏳ bloquant P2 |
| Stage 1 : parité waveform + TTFA | ⏳ P3 |

Pour servir le **modèle finetuné FR + tool calling** : exporter avec 
`training/export_checkpoint.py --mode full`, pointer `BASE` (cellule 6) sur cet export, 
le ratio interleaved calibré est lu depuis son `config.json`.

In [ ]:
import time, numpy as np, torch
from IPython.display import Audio, display
from vllm import SamplingParams
from vllm_omni_lfm2_audio.constants import (
    AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID, IM_END_TOKEN_ID)

SYS = "You are a helpful assistant that can respond with interleaved text and audio. Use the following format:\n" 
history = []  # (role, texte) — on ne garde que le texte en contexte (pas les placeholders)

def _render(h):
    s = "<|startoftext|><|im_start|>system\n" + SYS + "<|im_end|>\n"
    for role, txt in h:
        s += f"<|im_start|>{role}\n{txt}<|im_end|>\n"
    return tok(s + "<|im_start|>assistant\n", add_special_tokens=False).input_ids

def _wave(x):
    if isinstance(x, dict):
        for k in ('model_outputs', 'audio', 'waveform', 'wav'):
            if k in x: return _wave(x[k])
    if isinstance(x, torch.Tensor): return x.detach().float().cpu().numpy().reshape(-1)
    if isinstance(x, np.ndarray): return x.reshape(-1)
    return None

def say(user_text, max_tokens=256):
    history.append(("user", user_text))
    ids = _render(history)
    sp0 = SamplingParams(temperature=0.0, max_tokens=max_tokens, stop_token_ids=[IM_END_TOKEN_ID])
    sp1 = SamplingParams(max_tokens=1, detokenize=False)
    t0 = time.time()
    outs = omni.generate({"prompt_token_ids": ids}, [sp0, sp1])
    dt = time.time() - t0
    text, wav, nf = "", None, 0
    for o in outs:
        ro = o.request_output
        if o.final_output_type == "text" and ro and ro.outputs:
            toks = list(ro.outputs[0].token_ids)
            nf = sum(t in (AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID) for t in toks)
            text = ro.outputs[0].text
        elif o.final_output_type == "audio":
            wav = _wave(getattr(ro, "multimodal_output", None) or getattr(o, "multimodal_output", None))
    history.append(("assistant", text))
    print(f"🤖 {text}\n   {nf} frames · {nf/12.5:.1f}s audio · généré en {dt:.1f}s")
    if wav is not None and wav.size:
        display(Audio(wav, rate=24000))
    return text

print("Dialogue — message vide pour quitter.")
while True:
    u = input("👤 ").strip()
    if not u: break
    say(u)


Dialogue — message vide pour quitter.
INFO 06-11 14:40:35 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-0 [rep-0] add request: 0_5058f2fe-b7ad-46d7-b435-61d53793d71f
INFO 06-11 14:40:35 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-1 [rep-0] add request: 0_5058f2fe-b7ad-46d7-b435-61d53793d71f


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s]

🤖 Hello! How can I assist you today?
   45 frames · 3.6s audio · généré en 4.0s


INFO 06-11 14:41:03 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-0 [rep-0] add request: 0_f4d76dad-f5e3-4c3c-bf31-d49256c3b585
INFO 06-11 14:41:03 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-1 [rep-0] add request: 0_f4d76dad-f5e3-4c3c-bf31-d49256c3b585


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s]

🤖 I don’t have live internet access, so I can’t
   14 frames · 1.1s audio · généré en 1.3s


In [1]:
# === Référence liquid-audio (vérité terrain, SANS vLLM) ===
# Runtime frais + GPU. C'est la sortie "correcte" attendue du modèle de base.
import torch
from IPython.display import Audio, display
from liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState

model = LFM2AudioModel.from_pretrained("LiquidAI/LFM2.5-Audio-1.5B", device="cuda").eval()
proc  = LFM2AudioProcessor.from_pretrained("LiquidAI/LFM2.5-Audio-1.5B", device="cuda")

def ref_say(user_text, max_new_tokens=256):
    chat = ChatState(proc)
    chat.new_turn("system"); chat.add_text("Respond with interleaved text and audio."); chat.end_turn()
    chat.new_turn("user");   chat.add_text(user_text); chat.end_turn()
    chat.new_turn("assistant")
    text_ids, frames = [], []
    with torch.no_grad():
        for t in model.generate_interleaved(**chat, max_new_tokens=max_new_tokens):
            (text_ids if t.numel() == 1 else frames).append(t.detach().cpu())
    txt = proc.text.decode([int(x) for x in text_ids])
    print("🤖", txt, "|", len(frames), "frames")
    keep = [f.flatten() for f in frames if int(f.flatten()[0]) != 2048]
    if keep:
        fr = torch.stack(keep, dim=1).cuda()          # (8, T)
        wav = proc.decode(fr.unsqueeze(0))            # (1, 8, T) -> waveform
        display(Audio(wav.float().cpu().numpy().reshape(-1), rate=24000))

ref_say("Hello, who are you?")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

🤖 I’m Liquid Lili, an audio‑first assistant built by Liquid AI that can hear or read your input and respond with speech or text. How can I help you today?<|text_end|> | 131 frames


In [11]:

def ref_say(user_text, max_new_tokens=256):
    chat = ChatState(proc)
    chat.new_turn("system"); chat.add_text("Respond with interleaved text and audio."); chat.end_turn()
    chat.new_turn("user");   chat.add_text(user_text); chat.end_turn()
    chat.new_turn("assistant")
    text_ids, frames = [], []
    with torch.no_grad():
        for t in model.generate_interleaved(**chat, max_new_tokens=max_new_tokens):
            (text_ids if t.numel() == 1 else frames).append(t.detach().cpu())
    txt = proc.text.decode([int(x) for x in text_ids])
    print("🤖", txt, "|", len(frames), "frames")
    keep = [f.flatten() for f in frames if int(f.flatten()[0]) != 2048]
    if keep:
        fr = torch.stack(keep, dim=1).cuda()          # (8, T)
        wav = proc.decode(fr.unsqueeze(0))            # (1, 8, T) -> waveform
        display(Audio(wav.float().cpu().numpy().reshape(-1), rate=24000))

ref_say("Hello, who are you?")


🤖 I’m Liquid Lili, an audio‑first assistant built by Liquid AI that can hear or read your input and respond with speech or text. How can I help you today?<|text_end|> | 131 frames


In [16]:
import time
from IPython.display import Audio, display
#⚠️ nécessite un engine relancé avec async_chunk=True :
#   omni = Omni(model=OMNI, dtype='bfloat16', enforce_eager=True,
#               gpu_memory_utilization=0.42, async_scheduling=False,
#               async_chunk=True, stage_init_timeout=1200)

prompt_ids = build_prompt_ids("Bonjour, qui es-tu ?")
sp0 = SamplingParams(temperature=0.0, max_tokens=256, stop_token_ids=[IM_END_TOKEN_ID])
sp1 = SamplingParams(max_tokens=1, detokenize=False)

t0 = time.time(); ttfa = None; chunks = []
for out in omni.generate({'prompt_token_ids': prompt_ids}, [sp0, sp1], py_generator=True):
    if out.final_output_type == 'audio':
        w = to_waveform(getattr(out.request_output, 'multimodal_output', None))
        if w is not None and w.size:
            if ttfa is None:
                ttfa = time.time() - t0
                print(f"⏱️ TTFA = {ttfa*1000:.0f} ms")
            chunks.append(w)
print(f"{len(chunks)} chunks · total {time.time()-t0:.1f}s")
if chunks:
    import numpy as np
    display(Audio(np.concatenate(chunks), rate=24000))


INFO 06-11 15:11:22 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-0 [rep-0] add request: 0_fd218b8c-cd05-47f5-bb6b-4323cb17c084
INFO 06-11 15:11:22 [stage_engine_core_client.py:295] [StageEngineCoreClient] stage-1 [rep-0] add request: 0_fd218b8c-cd05-47f5-bb6b-4323cb17c084


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 06-11 15:11:29 [omni_base.py:584] [Omni] Shutting down
INFO 06-11 15:11:29 [async_omni_engine.py:2446] [AsyncOmniEngine] Shutting down Orchestrator
INFO 06-11 15:11:29 [orchestrator.py:435] [Orchestrator] Received shutdown signal
INFO 06-11 15:11:29 [orchestrator.py:1626] [Orchestrator] Shutting down all 2 client(s)
INFO 06-11 15:11:32 [stage_pool.py:1167] [StagePool] Stage 0 replica 0 shut down
INFO 06-11 15:11:35 [stage_pool.py:1167] [StagePool] Stage 1 replica 0 shut down
0 chunks · total 13.0s
